# Lincolnshire Adult Social Care: Data Pipeline

This notebook cleans four public data sources (ONS population estimates, ONS
population projections, DHSC ASCOF outcomes, CQC care directory) and loads
them into a local SQLite database for joining and aggregation, ready for
Power BI.

**Pipeline: Python (clean) -> SQL (join/aggregate) -> Power BI (model + visualise)**

Sources:
1. ONS mid-year population estimates 2024, single year of age (Lincolnshire + districts)
2. ONS 2022-based subnational population projections (Lincolnshire, to 2047)
3. ASCOF 2024/25 outcomes (DHSC / NHS England), council-level
4. CQC care directory with ratings (national file, filtered to Lincolnshire)

#Mount Drive, set paths

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from pathlib import Path
import pandas as pd
import sqlite3

#Project location in Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/lincolnshire_adult_care_project/project")

RAW = PROJECT_ROOT / "raw"
CLEAN = PROJECT_ROOT / "clean"
SQL = PROJECT_ROOT / "sql"
DB_PATH = PROJECT_ROOT / "lincolnshire_adult_care.db"
CLEAN.mkdir(exist_ok=True)

LINCS_CODE = "E10000019"

## 1. ONS population estimates 2024

Compute the 65+ share for Lincolnshire and its seven districts from single-year-of-age data.

In [11]:
def clean_population():
    df = pd.read_csv(RAW / "ons_population_2024.csv")

    def age_to_num(label):
        if label == "All Ages":
            return None
        if label == "Aged 90+":
            return 90
        return int(label.replace("Age ", ""))

    df["age_n"] = df["C_AGE_NAME"].apply(age_to_num)
    totals = df[df["GENDER_NAME"] == "Total"].copy()

    all_ages = totals[totals["C_AGE_NAME"] == "All Ages"][
        ["GEOGRAPHY_NAME", "GEOGRAPHY_CODE", "DATE_NAME", "OBS_VALUE"]
    ].rename(columns={"OBS_VALUE": "total_population"})

    age_rows = totals[totals["age_n"].notna()].copy()
    pop_65plus = (
        age_rows[age_rows["age_n"] >= 65]
        .groupby(["GEOGRAPHY_NAME", "GEOGRAPHY_CODE", "DATE_NAME"])["OBS_VALUE"]
        .sum()
        .reset_index()
        .rename(columns={"OBS_VALUE": "population_65_plus"})
    )

    out = all_ages.merge(pop_65plus, on=["GEOGRAPHY_NAME", "GEOGRAPHY_CODE", "DATE_NAME"])
    out["share_65_plus_pct"] = round(100 * out["population_65_plus"] / out["total_population"], 1)
    out = out.rename(columns={"GEOGRAPHY_NAME": "area_name", "GEOGRAPHY_CODE": "area_code", "DATE_NAME": "year"})
    out = out.sort_values("area_name")
    out.to_csv(CLEAN / "population_2024_by_area.csv", index=False)
    return out

pop_df = clean_population()
pop_df

,area_name,area_code,year,total_population,population_65_plus,share_65_plus_pct
1,Boston,E07000136,2024,71080,15067,21.2
2,East Lindsey,E07000137,2024,145183,45736,31.5
3,Lincoln,E07000138,2024,105114,15875,15.1
0,Lincolnshire,E10000019,2024,789502,190608,24.1
4,North Kesteven,E07000139,2024,122468,29277,23.9
5,South Holland,E07000140,2024,99298,24029,24.2
6,South Kesteven,E07000141,2024,147151,35105,23.9
7,West Lindsey,E07000142,2024,99208,25519,25.7


Lincolnshire's 65+ share (24.1%) sits well above every comparator except East Lindsey, and above the England average (18.5%).

## 2. ONS population projections, 2022-2047

The 65+ share is not just higher than average, it is projected to keep rising.

In [12]:
def clean_projections():
    df = pd.read_excel(RAW / "ons_population_projections.xls", engine="xlrd", sheet_name="Persons", header=3)
    lincs = df[df["CODE"] == LINCS_CODE].set_index("AGE GROUP")

    older_groups = ["65-69", "70-74", "75-79", "80-84", "85-89", "90+"]
    year_cols = [c for c in df.columns if isinstance(c, (int, float))]

    rows = []
    for y in year_cols:
        year = int(y)
        total = lincs.loc["All ages", y]
        pop65 = lincs.loc[older_groups, y].sum()
        rows.append({
            "area_name": "Lincolnshire", "area_code": LINCS_CODE, "year": year,
            "total_population": round(total), "population_65_plus": round(pop65),
            "share_65_plus_pct": round(100 * pop65 / total, 1),
        })
    out = pd.DataFrame(rows).sort_values("year")
    out.to_csv(CLEAN / "population_projection_lincolnshire.csv", index=False)
    return out

proj_df = clean_projections()
proj_df[proj_df.year.isin([2022, 2025, 2030, 2035, 2040])]

,area_name,area_code,year,total_population,population_65_plus,share_65_plus_pct
0,Lincolnshire,E10000019,2022,775864,184313,23.8
3,Lincolnshire,E10000019,2025,795680,194060,24.4
8,Lincolnshire,E10000019,2030,814710,217390,26.7
13,Lincolnshire,E10000019,2035,831116,238846,28.7
18,Lincolnshire,E10000019,2040,845331,251695,29.8


## 3. ASCOF 2024/25 outcomes

Each ASCOF measure lives on its own sheet with a different header offset. We pull the measures relevant to the Performance and Intelligence Officer role's person specification, for Lincolnshire and its statistical neighbours.

In [13]:
ASCOF_SHEETS = {
    "Table_2a": {"header": 6, "measure": "2A", "label": "Reablement: no further support needed", "outcome_col": "2024 to 2025 - outcome"},
    "Table_2b": {"header": 6, "measure": "2B", "label": "Admissions to care, 18-64 (per 100k)", "outcome_col": "2024 to 2025 - outcome"},
    "Table_2c": {"header": 6, "measure": "2C", "label": "Admissions to care, 65+ (per 100k)", "outcome_col": "2024 to 2025 - outcome"},
    "Table_2d(1)": {"header": 6, "measure": "2D1", "label": "Still at home 12wks post-discharge (65+)", "outcome_col": "2024 to 2025 - outcome"},
    "Table_4b": {"header": 6, "measure": "4B", "label": "Safeguarding: risk reduced or removed", "outcome_col": "2024 to 2025 - outcome"},
    "Table_6a": {"header": 4, "measure": "6A", "label": "Care workforce turnover", "outcome_col": "2024 to 2025 - outcome"},
}

# Note: Cumbria is excluded as a comparator. It was abolished in April 2023
# and replaced by two new unitary authorities (Westmorland and Furness,
# Cumberland), so it no longer appears as a single area in current ASCOF data.
COMPARATORS = [
    "Lincolnshire", "England", "Norfolk", "Suffolk", "Devon", "Somerset",
    "Staffordshire", "Worcestershire", "Nottinghamshire", "East Midlands",
]

def clean_ascof():
    all_rows = []
    for sheet, cfg in ASCOF_SHEETS.items():
        df = pd.read_excel(RAW / "ascof_2024_25.ods", engine="odf", sheet_name=sheet, header=cfg["header"])
        la_col = [c for c in df.columns if "Local authority" in str(c) and "code" not in str(c)][0]
        df = df[df[la_col].isin(COMPARATORS)]
        for _, r in df.iterrows():
            all_rows.append({
                "measure_code": cfg["measure"], "measure_label": cfg["label"],
                "area_name": r[la_col], "year": "2024 to 2025", "value": r[cfg["outcome_col"]],
            })
    out = pd.DataFrame(all_rows)
    out = out[pd.to_numeric(out["value"], errors="coerce").notna()]
    out["value"] = out["value"].astype(float)
    out.to_csv(CLEAN / "ascof_outcomes_2024_25.csv", index=False)
    return out

ascof_df = clean_ascof()
ascof_df[ascof_df.area_name == "Lincolnshire"]

,measure_code,measure_label,area_name,year,value
4,2A,Reablement: no further support needed,Lincolnshire,2024 to 2025,98.00
14,2B,"Admissions to care, 18-64 (per 100k)",Lincolnshire,2024 to 2025,22.80
24,2C,"Admissions to care, 65+ (per 100k)",Lincolnshire,2024 to 2025,588.10
34,2D1,Still at home 12wks post-discharge (65+),Lincolnshire,2024 to 2025,63.24
44,4B,Safeguarding: risk reduced or removed,Lincolnshire,2024 to 2025,93.80
54,6A,Care workforce turnover,Lincolnshire,2024 to 2025,30.20


**Reading this honestly:** Lincolnshire outperforms England on reablement (98.0% vs 77.1%), safeguarding (93.8% vs 91.2%) and discharge outcomes (63.2% vs 60.7%). It underperforms on workforce turnover (30.2% vs 23.7%) and working-age admissions (22.8 vs 17.0 per 100k). Several 2024/25 measures are calculated from the new Client Level Dataset and are official statistics in development, so should not be compared to pre-2024 SALT figures.

## 4. CQC care directory

The national file is ~110MB, so it was pre-filtered down to Lincolnshire care homes only before being added to this project (`cqc_lincolnshire_prefiltered.csv`). The full national file is not committed to this repository.

In [14]:
def clean_cqc():
    df = pd.read_csv(RAW / "cqc_lincolnshire_prefiltered.csv")
    homes = df[(df["Care Home?"] == "Y") & (df["Domain"] == "Overall")]
    homes = homes.drop_duplicates(subset="Location ID")
    homes = homes[["Location Name", "Location Post Code", "Latest Rating", "Provider Name"]].rename(
        columns={"Location Name": "care_home_name", "Location Post Code": "postcode",
                 "Latest Rating": "rating", "Provider Name": "provider_name"}
    )
    homes.to_csv(CLEAN / "cqc_care_homes_lincolnshire.csv", index=False)
    return homes

cqc_df = clean_cqc()
print(f"{len(cqc_df)} Lincolnshire care homes")
cqc_df["rating"].value_counts()

262 Lincolnshire care homes


,count
rating,
Good,184
Requires improvement,56
Outstanding,16
Not Rated,3
Inadequate,3


## 5. Load into SQLite and build the dashboard tables

The four cleaned tables are loaded into a local SQLite database. SQL then joins and aggregates them into three flat tables, one per dashboard page, so Power BI reads a single clean table per page rather than repeating the join logic in DAX.

In [15]:
if DB_PATH.exists():
    DB_PATH.unlink()
conn = sqlite3.connect(DB_PATH)

with open(SQL / "01_create_tables.sql") as f:
    conn.executescript(f.read())

TABLE_MAP = {
    "population_2024_by_area.csv": "population_by_area",
    "population_projection_lincolnshire.csv": "population_projection",
    "ascof_outcomes_2024_25.csv": "ascof_outcomes",
    "cqc_care_homes_lincolnshire.csv": "cqc_care_homes",
}
for csv_name, table_name in TABLE_MAP.items():
    pd.read_csv(CLEAN / csv_name).to_sql(table_name, conn, if_exists="append", index=False)

with open(SQL / "02_build_dashboard_tables.sql") as f:
    conn.executescript(f.read())

conn.commit()
print("Database built:", DB_PATH)

Database built: /content/drive/MyDrive/lincolnshire_adult_care_project/project/lincolnshire_adult_care.db


In [16]:
# Export the three dashboard-ready tables as CSV for direct Power BI import
for t in ["dashboard_demand", "dashboard_outcomes", "dashboard_providers", "dashboard_providers_summary"]:
    df = pd.read_sql(f"SELECT * FROM {t}", conn)
    df.to_csv(CLEAN / f"{t}.csv", index=False)
    print(f"{t}: {len(df)} rows")

conn.close()

dashboard_demand: 14 rows
dashboard_outcomes: 54 rows
dashboard_providers: 259 rows
dashboard_providers_summary: 4 rows


## 6. Quick sanity check

Before handing these tables to Power BI, confirm the join preserved every row and the England benchmark column populated correctly.

In [17]:
check = pd.read_csv(CLEAN / "dashboard_outcomes.csv")
assert check["england_value"].notna().all(), "Missing England benchmark on some rows"
assert len(check) == 54, f"Expected 54 rows, got {len(check)}"
print("Outcomes table check passed:", check.shape)

check2 = pd.read_csv(CLEAN / "dashboard_providers_summary.csv")
print("\nProvider ratings summary:")
check2

Outcomes table check passed: (54, 6)

Provider ratings summary:


,rating,home_count,pct_of_total
0,Good,184,71.0
1,Requires improvement,56,21.6
2,Outstanding,16,6.2
3,Inadequate,3,1.2
